# Session 3 — SciPy: adding a statistical layer

**Plan for today**
1. Cleaning, then fitting distributions
2. Hypothesis testing
3. Correlation & optimization

Each section has **core** exercises (everyone should complete these) and <span style="color:red">**advanced / optional**</span> exercises — you do **not** need to answer them.

Today's dataset is the **Pima Indians Diabetes Dataset** (`diabetes.csv`) — 768 real, anonymized
patient records, with 8 diagnostic measurements and a binary `Outcome` (1 = tested positive for
diabetes, 0 = negative). Make sure `diabetes.csv` is in the same folder as this notebook.

**A running theme today**: real data rarely fits a textbook distribution perfectly, and formal
statistical tests are often stricter judges than a histogram "looks like." That's not a flaw in the
exercises — learning to read an *inconclusive* or *messy* result honestly is as important as reading
a clean one. This dataset in particular has a well-known real-world quirk we'll deal with directly
in the first exercise.


In [ ]:
# Run this first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize

sns.set_theme(style="whitegrid")
np.random.seed(42)


## Setup — loading and cleaning the data

Load `diabetes.csv`. The columns are: `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`,
`Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`, and `Outcome` (1 = diabetic, 0 = not).


In [ ]:
diabetes_raw = pd.read_csv("diabetes.csv")
print(diabetes_raw.shape)
diabetes_raw.head()


### Exercise 0.1 (core) — The zeros that aren't really zeros
Take a look at the data and answer the usual questions:

1. How many rows and columns are there?
2. What are the column names?
3. What data type does each column have?
4. Are there missing values?

In [ ]:
diabetes_raw.info()

Now run the cell below and look closely at the minimum values. How do you interpret them?

In [ ]:
print(
    diabetes_raw[
        ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
    ].describe()
)


<details>
<summary>A blood pressure of <b>0</b>? A BMI of <b>0</b>? What does that mean?</summary>
Those aren't real measurements. They're <b>physically impossible</b>, which means they almost certainly represent <i>missing</i> data that got encoded as <code>0</code> instead of left blank. 

This is actually a well-documented data-quality issue of this specific dataset, a good reminder that "no missing values" (`.isna().sum()` on
`diabetes_raw` would show all zeros!) doesn't mean a column is actually complete. Missing data can hide in plain sight as a valid-looking number.
</details>

It is generally accepted that zero values were used as placeholders for missing measurements for (`Glucose`, `BloodPressure`, `SkinThickness`,
`Insulin`, `BMI`).

Replace `0` with `NaN` in the five affected columns (`Glucose`, `BloodPressure`, `SkinThickness`,
`Insulin`, `BMI`), but **not** `Pregnancies` (zero is a completely valid number of pregnancies) and
**not** `Outcome` (zero validly means "not diabetic"). Store the result in `diabetes` and print the
missing-value counts.

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

# TODO: copy diabetes_raw into `diabetes`, then replace 0 with NaN in zero_as_missing columns only
diabetes = None

print(diabetes.isna().sum())


In [ ]:
assert diabetes["BloodPressure"].isna().sum() == 35
assert diabetes["Insulin"].isna().sum() == 374
assert diabetes["Pregnancies"].isna().sum() == 0  # zero pregnancies is valid, untouched
assert diabetes["Outcome"].isna().sum() == 0       # zero outcome is valid, untouched
print("Looks good! Insulin alone is missing for nearly half the patients -- keep that in mind below.")


## 1. Fitting distributions

### Exercise 1.1 (core) — Fit a normal distribution to Glucose

Drop missing values from `Glucose`, fit a normal distribution (`scipy.stats.norm.fit`, returns
`(mean, std)`), and plot the histogram of the actual data with the fitted normal PDF overlaid.

Then run a Shapiro-Wilk test on the same data and compare what the plot *looks like* against what
the test formally concludes.


In [ ]:
glucose = diabetes["Glucose"].dropna()

# TODO: fit a normal distribution to glucose
mu, sigma = None, None  # stats.norm.fit() returns (mean, std)

x = np.linspace(glucose.min(), glucose.max(), 200)
plt.figure(figsize=(8, 5))
sns.histplot(glucose, stat="density", bins=25, alpha=0.6, label="actual")
plt.plot(x, stats.norm.pdf(x, mu, sigma), label="fitted normal", linewidth=2)
plt.legend()
plt.title("Glucose: actual vs. fitted normal")
plt.show()

# TODO: run a Shapiro-Wilk test on glucose
shapiro_stat, shapiro_p = None, None  # stats.shapiro() returns (statistic, p_value)
print(f"Fitted: mu={mu:.1f}, sigma={sigma:.1f}")
print(f"Shapiro: statistic={shapiro_stat:.4f}, p={shapiro_p:.2e}")


### Exercise 1.2 (core) — Fit a Poisson distribution to Pregnancies

`Pregnancies` is a genuine **count** variable — a natural fit for testing a Poisson model. Fit
lambda (just the sample mean, no need to drop anything here — `Pregnancies` has no missing-as-zero
issue, unlike the other columns), plot actual vs. fitted Poisson PMF, and compare the mean to the
variance.

**Why compare mean and variance?** A defining property of a true Poisson distribution is that its
mean *equals* its variance. If the real variance is a lot bigger than the mean, the data is more
spread out than Poisson expects — a pattern called **overdispersion**, and a very common finding in
real count data (Poisson assumes every event is independent and equally likely at every moment,
which is rarely exactly true for something like "number of pregnancies").


In [ ]:
pregnancies = diabetes["Pregnancies"]

# TODO: fit lambda (sample mean) and plot actual vs. fitted Poisson PMF
lam = None

k = np.arange(0, pregnancies.max() + 1)
plt.figure(figsize=(8, 5))
sns.histplot(pregnancies, stat="density", bins=np.arange(-0.5, pregnancies.max() + 1.5),
             alpha=0.6, label="actual")
plt.plot(k, stats.poisson.pmf(k, lam), "o-", label="fitted Poisson", linewidth=2)
plt.legend()
plt.title("Pregnancies: actual vs. fitted Poisson")
plt.show()

print(f"Fitted lambda: {lam:.2f}")
print(f"Mean: {pregnancies.mean():.2f}, Variance: {pregnancies.var():.2f}")


### Exercise 1.3 (<span style="color:red">advanced / optional</span>) — Compare distributions

`Insulin` is heavily right-skewed and strictly non-negative (a measurement can't be negative) —
a good candidate for lognormal or gamma instead of normal.

Fit **normal**, **lognormal**, and **gamma** to `Insulin` (dropping missing values — remember,
nearly half of this column is missing), using the frozen-distribution + `kstest` pattern from before.

**Important note**: `Insulin` values cannot be negative so we need to constrain the *floor* for **both** lognormal *and* gamma this time.

In `scipy`, prefixing a fit parameter with `f` tells `.fit()` to hold it fixed rather than estimate it, so
`dist.fit(data, floc=0)` is how you fix the location (floor) at exactly `0`.
<details>
<summary>Try fitting gamma *without* that constraint first, look at the fitted parameters and the resulting KS statistic, then compare against the constrained version.</summary>

You'll likely see something surprising: the *unconstrained* fit is dramatically **worse** (a KS
statistic near 1, essentially a complete rejection), not deceptively better. With `loc` free, the
optimizer can converge to a degenerate solution — when the fitted shape parameter is under 1,
gamma's density spikes sharply as `x` approaches `loc`, so MLE fitting can "cheat" by parking `loc`
right at (or just under) the sample minimum to inflate the likelihood there, at the cost of badly
describing everything else. Constraining `loc=0` — the one value that's actually physically
justified, since `Insulin` can't be negative — avoids this pathology entirely and produces a far
better, sensible fit. The general rule still holds: when you know a hard physical floor, constrain
`loc` to it rather than letting the optimizer wander.
</details>

In [ ]:
insulin = diabetes["Insulin"].dropna()

# TODO: fit gamma WITHOUT constraining loc -- print the params and KS result
params_unconstrained = None

# TODO: fit norm, lognorm (floc=0), and gamma (floc=0); compute KS statistic for each using a
# frozen distribution's .cdf
results = {}

for dist_name in ["norm", "lognorm", "gamma"]:
    pass  # fit (floc=0 for lognorm/gamma), freeze, run kstest, store in results

for name, res in results.items():
    print(name, res)

best = None  # TODO: name of the distribution with the smallest ks_stat
print("Best fit:", best)


## 2. Hypothesis testing

### Exercise 2.1 (core) — t-test: is Glucose different between diabetic and non-diabetic patients?

Split `Glucose` (missing values dropped) by `Outcome`, and run an independent two-sample t-test
(`scipy.stats.ttest_ind`) to check whether the difference in mean glucose is statistically
significant.

Using `alpha = 0.05`: is the difference significant? Given that elevated glucose is literally part
of how diabetes is diagnosed, what result would you *expect* here before running it?


In [ ]:
glucose_data = diabetes.dropna(subset=["Glucose"])
g0 = glucose_data.loc[glucose_data["Outcome"] == 0, "Glucose"]
g1 = glucose_data.loc[glucose_data["Outcome"] == 1, "Glucose"]

# TODO: run the t-test
t_stat, p_value = None, None  # stats.ttest_ind() returns (statistic, p_value)

alpha = 0.05
print(f"Non-diabetic mean: {g0.mean():.1f}, Diabetic mean: {g1.mean():.1f}")
print(f"t = {t_stat:.3f}, p = {p_value:.2e}")
print("Significant?", p_value < alpha)


In [ ]:
assert p_value < 0.001
print("Looks good!")


### Exercise 2.2 (core) — Chi-square: is BMI category associated with diabetes?

Bucket `BMI` (missing values dropped) into the standard clinical categories: Underweight (< 18.5),
Normal (18.5–25), Overweight (25–30), Obese (30+) — using `pd.cut`. Build a contingency table of
this category against `Outcome`, and run `scipy.stats.chi2_contingency`.

**Before trusting the result**, check the test's assumption: `chi2_contingency` returns the
*expected* counts as its 4th value. The chi-square test is only reliable when expected counts are
reasonably large (a common rule of thumb: **at least 5** in every cell). Print the expected table —
if any cell is too small, **merge Underweight into Normal** (there are very few underweight patients
in this dataset) and rerun before drawing a conclusion.


In [ ]:
bmi_data = diabetes.dropna(subset=["BMI"]).copy()

# TODO 1: bucket BMI into 4 categories with pd.cut
bins = [0, 18.5, 25, 30, 100]
labels = ["Underweight", "Normal", "Overweight", "Obese"]
bmi_data["bmi_category"] = None

# TODO 2: contingency table + chi2_contingency
contingency = None
chi2_stat, p_value, dof, expected = None, None, None, None  # chi2_contingency() returns these 4 values

print(contingency)
print("Expected counts:\n", expected)
print(f"chi2 = {chi2_stat:.2f}, p = {p_value:.2e}, dof = {dof}")

# TODO 3: if any expected count is below 5, merge Underweight into Normal and rerun


In [ ]:
assert expected2.min() >= 5
assert p_value2 < 0.001
print("Looks good! BMI category and Outcome are significantly associated, now on a valid test.")


### Exercise 2.3 (<span style="color:red">advanced / optional</span>) — Check normality first, then pick the right test

Write `compare_two_groups(group1, group2, alpha=0.05)`: run `stats.shapiro` on **both** groups;
if both pass (p > alpha), run a t-test; otherwise run `stats.mannwhitneyu`. Return
`{"test_used": ..., "statistic": ..., "p_value": ...}`.

Apply it to `Insulin` split by `Outcome` — you already know from 1.3 that `Insulin` is heavily
skewed, so this is a real test of the branching logic, not a toy case.


In [ ]:
def compare_two_groups(group1, group2, alpha=0.05):
    # TODO: check normality of both groups, branch to ttest_ind or mannwhitneyu accordingly
    pass

insulin_data = diabetes.dropna(subset=["Insulin"])
i0 = insulin_data.loc[insulin_data["Outcome"] == 0, "Insulin"]
i1 = insulin_data.loc[insulin_data["Outcome"] == 1, "Insulin"]

result = compare_two_groups(i0, i1)
print(result)


## 3. Correlation & optimization

### Exercise 3.1 (core) — Pearson correlation, with significance

Compute the Pearson correlation (`scipy.stats.pearsonr`) between:
1. `Age` and `Pregnancies`
2. `Glucose` and `BMI`

(Drop missing values pairwise for each pair — `.dropna()` on just the two columns involved, since
different columns have different missing patterns.)

Both should come back statistically significant given how much data we have — but look closely at
the **size** of `r` in each case, not just the p-value. A relationship can be statistically
significant (real, not noise) while still being practically weak. With a large enough sample, even
a small, not-very-useful correlation can produce a tiny p-value.


In [ ]:
# TODO: pearsonr for both pairs (drop NaN pairwise for each)
pair1 = diabetes[["Age", "Pregnancies"]].dropna()
r1, p1 = None, None  # stats.pearsonr() returns (r, p_value)

pair2 = diabetes[["Glucose", "BMI"]].dropna()
r2, p2 = None, None

print(f"Age vs Pregnancies: r={r1:.3f}, p={p1:.2e}, n={len(pair1)}")
print(f"Glucose vs BMI:     r={r2:.3f}, p={p2:.2e}, n={len(pair2)}")


### Exercise 3.2 (core) — Spearman as a robustness check

Recompute both pairs from 3.1 using **Spearman's rank correlation**. Spearman only cares about
monotonic relationships and is less sensitive to outliers than Pearson — a useful cross-check
whenever a relationship might not be perfectly linear.


In [ ]:
# TODO: spearmanr for both pairs
rho1, sp1 = None, None  # stats.spearmanr() returns (rho, p_value)
rho2, sp2 = None, None

print(f"Age vs Pregnancies: rho={rho1:.3f}, p={sp1:.2e}")
print(f"Glucose vs BMI:     rho={rho2:.3f}, p={sp2:.2e}")


### Exercise 3.3 (<span style="color:red">advanced / optional</span>) — Minimize a cost function

A clinic orders glucose test strips in batches. Larger batches mean fewer orders (lower ordering
cost) but more strips sitting in storage at once (higher holding cost). Given:

```
total_cost(x) = 8000 / x + 1.5 * x
```

where `x` is the batch size, find the `x` that **minimizes** total cost, using
`scipy.optimize.minimize_scalar` with `bounds=(1, 10000)` and `method="bounded"`.


In [ ]:
def total_cost(x):
    return 8000 / x + 1.5 * x

# TODO: minimize total_cost over x in [1, 10000]
result = None

print(f"Optimal batch size: {result.x:.1f}, minimum cost: {result.fun:.1f}")


In [ ]:
assert abs(result.x - 73.03) < 1
print("Looks good!")


### Exercise 3.4 (<span style="color:red">advanced / optional</span>) — Multi-variable optimization with a constraint

A public health department has a $100,000 outreach budget (in $1000 units, `x + y = 100`) to split
between two diabetes-screening programs, each with **diminishing returns** (more funding helps, but
with less extra benefit each time):

```
expected_screenings(x, y) = 5*x - 0.03*x^2 + 8*y - 0.02*y^2
```

Find the split `(x, y)` that **maximizes** expected screenings, subject to `x + y = 100` and
`x, y >= 0`, using `scipy.optimize.minimize` (minimize the *negative*), `method="SLSQP"`, an
equality constraint, and bounds.


In [ ]:
def neg_expected_screenings(vars):
    x, y = vars
    return -(5*x - 0.03*x**2 + 8*y - 0.02*y**2)

# TODO: set up the constraint (x + y == 100) and bounds (0 <= x,y <= 100), then minimize
constraints = None
bounds = None

result = None

print(f"Optimal split: program A = {result.x[0]:.1f}, program B = {result.x[1]:.1f}")
print(f"Expected screenings: {-result.fun:.1f}")


---

**That's a wrap for the SciPy layer**: fitting real distributions, running the right hypothesis
test (and knowing why it's the right one), and checking correlation with significance -- all on
real data, with real, sometimes inconclusive results along the way.
